In [ ]:
!git clone https://github.com/Thinh-MaNU/Machine-learning-project-.git
%cd Machine-learning-project-
!pip install -r requirements.txt -q

In [ ]:
import wandb
wandb.login()

In [ ]:
import torch
import wandb
from src.datasets import (
    CORRUPTIONS_ORDER,
    get_gradual_sequence,
    load_corruption_dataloader,
    prepare_dataset_from_hf,
)
from src.utils import (
    init_wandb,
    load_config,
    load_source_model,
    log_corruption_metrics,
    set_seed,
)

# 1. Test nhanh hàm tạo chuỗi Gradual (Bảng 3)
gradual_seq = get_gradual_sequence(seed=42)
print(f"[Check Gradual] Tổng số chặng (15 corruptions x 9 levels): {len(gradual_seq)}")
assert len(gradual_seq) == 135

# 2. Hàm chạy kiểm tra Data + Model nguồn + Log W&B
def run_source_sanity_check(config_path: str):
    cfg = load_config(config_path)
    set_seed(cfg["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n=== Đang test {cfg['dataset'].upper()} trên {device} ===")

    data_root = prepare_dataset_from_hf(cfg["hf_repo_id"], cfg["data_dir"])
    model = load_source_model(
        cfg["model_name"], cfg["dataset"], cfg["threat_model"], cfg["model_dir"], device
    )
    model.eval()

    run = init_wandb(
        {**cfg, "method": "source_sanity_check"},
        run_name=f"{cfg['dataset']}-source-check",
        tags=["sanity-check", cfg["dataset"]],
    )
    print(f"Link W&B Run: {run.url}")

    errors = []
    for idx, corr in enumerate(CORRUPTIONS_ORDER, start=1):
        loader = load_corruption_dataloader(
            data_root, corr, cfg["severity"], cfg["batch_size"]
        )
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                preds = model(x).argmax(dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        err = (1.0 - correct / total) * 100.0
        errors.append(err)
        log_corruption_metrics(idx, corr, err)
        print(f"  {idx:02d}. {corr:<20s} | Error: {err:.2f}%")

    mean_err = sum(errors) / len(errors)
    wandb.summary["mean_error_rate"] = mean_err
    print(f"===> {cfg['dataset'].upper()} MEAN SOURCE ERROR: {mean_err:.2f}% <===")
    run.finish()

# Chạy kiểm tra cả CIFAR-10-C và CIFAR-100-C
run_source_sanity_check("configs/default.yaml")
run_source_sanity_check("configs/cifar100c.yaml")